In [ ]:
#!/usr/bin/env python3
"""
TinySSL: Distilling DINOv2 into Tiny Vision Models
===================================================
Self-contained Colab notebook.

Trains a ~3M-param student to match DINOv2 (22M) on linear probing,
using knowledge distillation + MIM-JEPA on frozen teacher features.

Runtime: ~45 min on Colab T4 GPU (AMP enabled).
"""

In [ ]:
# Cell 1: Install + Clear Caches
!pip install -q timm scikit-learn matplotlib seaborn pandas medmnist pillow
import shutil
from pathlib import Path
if Path("cache").exists():
    shutil.rmtree("cache")
print("Ready.")

In [ ]:
# Cell 2: Imports
import os, math, time, json, gc
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import timm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from PIL import Image as PILImage
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = torch.cuda.is_available()
print(f"Device: {device} | AMP: {AMP}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Cell 3: DINOv2 Teacher
DINO_DIM = 384

class DINOv2Teacher(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
        for p in self.net.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def forward(self, x):
        features = self.net.get_intermediate_layers(x, n=[9])
        last = features[-1]
        return {
            "cls_token": last[:, 0],
            "patch_tokens": last[:, 1:],
        }

In [ ]:
# Cell 4: Student Models
class TinySSLBase(nn.Module):
    def __init__(self, img_size=224, out_dim=256, stride=4):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, 256, kernel_size=3, stride=stride, padding=1)
        n_patches = (img_size // stride) ** 2
        self.proj = nn.Linear(256, out_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=out_dim, nhead=4, dim_feedforward=512, batch_first=True, dropout=0.1
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=4)
        self.cls_token = nn.Parameter(torch.randn(1, 1, out_dim) * 0.02)
        self.out_dim = out_dim
        self.n_patches = n_patches

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)
        x = self.proj(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.encoder(x)
        return {"cls": x[:, 0], "patches": x[:, 1:]}


class TinySSLTiny(nn.Module):
    def __init__(self, img_size=224, out_dim=128, stride=4):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, 128, kernel_size=3, stride=stride, padding=1)
        n_patches = (img_size // stride) ** 2
        self.proj = nn.Linear(128, out_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=out_dim, nhead=4, dim_feedforward=256, batch_first=True, dropout=0.1
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.out_dim = out_dim
        self.n_patches = n_patches

    def forward(self, x):
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)
        x = self.proj(x)
        x = self.encoder(x)
        cls = x.mean(dim=1)
        return {"cls": cls, "patches": x}


class TinySSLCNN(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        self.blocks = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.head = nn.Linear(256, out_dim)
        self.out_dim = out_dim
        self.n_patches = 196

    def forward(self, x):
        feat = self.blocks(x)
        cls = feat.mean(dim=(2, 3))
        cls = self.head(cls)
        patches = feat.flatten(2).transpose(1, 2)
        return {"cls": cls, "patches": patches}


def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

for name, cls in [("Base", TinySSLBase), ("Tiny", TinySSLTiny), ("CNN", TinySSLCNN)]:
    m = cls()
    print(f"TinySSL-{name}: {count_params(m):.2f}M params, {m.n_patches} patches")

In [ ]:
# Cell 5: Losses + Projection (trainable, per-model)
_proj_modules = []

def create_projection(D_s, D_t, device):
    proj = nn.Sequential(
        nn.Linear(D_s, D_t),
        nn.GELU(),
        nn.Linear(D_t, D_t),
    ).to(device)
    _proj_modules.append(proj)
    return proj


def clear_projections():
    _proj_modules.clear()


def distillation_loss(student_patches, teacher_patches, proj):
    s = F.normalize(student_patches, dim=-1)
    t = F.normalize(teacher_patches, dim=-1)
    p = F.normalize(proj(s).mean(dim=1), dim=-1)
    t_mean = t.mean(dim=1)
    return (1.0 - (p * t_mean).sum(-1)).mean()

In [ ]:
# Cell 6: Augmentations + Dataset Loading
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def get_augmentation(epoch):
    if epoch < 100:
        augs = [transforms.Resize(256), transforms.RandomHorizontalFlip(), transforms.RandomCrop(224, padding=4)]
    elif epoch < 150:
        augs = [
            transforms.Resize(256), transforms.RandomHorizontalFlip(), transforms.RandomCrop(224, padding=4),
            transforms.ColorJitter(0.4, 0.4, 0.4),
            transforms.GaussianBlur(23, (0.1, 2.0)),
        ]
    else:
        augs = [
            transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.8, 0.8, 0.8),
            transforms.GaussianBlur(23, (0.1, 2.0)),
            transforms.RandomSolarize(0.5),
        ]
    augs += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(augs)


def load_dataset_for_caching(name):
    t = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor()])
    if name == "flowers102":
        return datasets.Flowers102(root="data", split="train", download=True, transform=t)
    elif name == "oxford_pets":
        return datasets.OxfordIIITPet(root="data", download=True, transform=t)
    elif name == "eurosat":
        return datasets.EuroSAT(root="data", download=True, transform=t)
    elif name == "breastmnist":
        import medmnist
        return medmnist.BreastMNIST(root="data", split="train", download=True,
                                     transform=transforms.Compose([
                                         transforms.Resize(224),
                                         transforms.Grayscale(num_output_channels=3),
                                         transforms.ToTensor(),
                                     ]))
    raise ValueError(f"Unknown dataset: {name}")


def load_pil_images(name):
    if name == "flowers102":
        raw = datasets.Flowers102(root="data", split="train", download=True)
        return [raw[i][0] for i in range(len(raw))]
    elif name == "oxford_pets":
        raw = datasets.OxfordIIITPet(root="data", download=True)
        return [raw[i][0] for i in range(len(raw))]
    elif name == "eurosat":
        raw = datasets.EuroSAT(root="data", download=True)
        return [raw[i][0] for i in range(len(raw))]
    elif name == "breastmnist":
        import medmnist
        raw = medmnist.BreastMNIST(root="data", split="train", download=True)
        images = []
        for i in range(len(raw)):
            img = raw[i][0]
            arr = np.array(img)
            if arr.ndim == 2:
                arr = np.stack([arr]*3, axis=-1)
            img = PILImage.fromarray(arr).resize((224, 224))
            images.append(img)
        return images
    raise ValueError(f"Unknown dataset: {name}")


def load_eval_datasets(name):
    eval_t = transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    train_t = transforms.Compose([
        transforms.Resize(256), transforms.RandomCrop(224, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    if name == "flowers102":
        return (datasets.Flowers102(root="data", split="train", download=True, transform=train_t),
                datasets.Flowers102(root="data", split="test", download=True, transform=eval_t), 102)
    elif name == "oxford_pets":
        return (datasets.OxfordIIITPet(root="data", download=True, transform=train_t, split="trainval"),
                datasets.OxfordIIITPet(root="data", download=True, transform=eval_t, split="test"), 37)
    elif name == "eurosat":
        full_train = datasets.EuroSAT(root="data", download=True, transform=train_t)
        full_test = datasets.EuroSAT(root="data", download=True, transform=eval_t)
        n = len(full_train)
        n_train = int(0.8 * n)
        gen = torch.Generator().manual_seed(42)
        train_indices = torch.randperm(n, generator=gen)[:n_train].tolist()
        test_indices = torch.randperm(n, generator=gen)[n_train:].tolist()
        return (torch.utils.data.Subset(full_train, train_indices),
                torch.utils.data.Subset(full_test, test_indices), 10)
    elif name == "breastmnist":
        import medmnist
        t = transforms.Compose([transforms.Resize(224), transforms.Grayscale(num_output_channels=3),
                                 transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
        return (medmnist.BreastMNIST(root="data", split="train", download=True, transform=t),
                medmnist.BreastMNIST(root="data", split="test", download=True, transform=t), 2)
    raise ValueError(f"Unknown dataset: {name}")


DATASETS = ["flowers102", "oxford_pets", "eurosat", "breastmnist"]
DATASET_INFO = {"flowers102": 102, "oxford_pets": 37, "eurosat": 10, "breastmnist": 2}

In [ ]:
# Cell 7: Cache DINOv2 Features
def interp_patches(teacher_patches, target_n):
    B, _, D = teacher_patches.shape
    s = int(teacher_patches.shape[1] ** 0.5)
    t = teacher_patches.view(B, s, s, D).permute(0, 3, 1, 2)
    ts = int(target_n ** 0.5)
    t = F.interpolate(t, size=(ts, ts), mode="bilinear", align_corners=False)
    return t.permute(0, 2, 3, 1).reshape(B, target_n, D)


def cache_teacher_features(dataset_name, batch_size=64, max_samples=2000):
    cache_path = Path("cache") / f"{dataset_name}.pt"
    if cache_path.exists():
        print(f"  Cache exists, skipping")
        return cache_path
    Path("cache").mkdir(parents=True, exist_ok=True)
    teacher = DINOv2Teacher().to(device).eval()
    dataset = load_dataset_for_caching(dataset_name)
    n = min(len(dataset), max_samples)
    subset = torch.utils.data.Subset(dataset, list(range(n)))
    loader = DataLoader(subset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    chunk_dir = Path("cache") / f"{dataset_name}_chunks"
    chunk_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with torch.no_grad():
        for i, (imgs, _labels) in enumerate(loader):
            imgs = imgs.to(device, non_blocking=True)
            out = teacher(imgs)
            torch.save({
                "cls": out["cls_token"].cpu().half(),
                "patches": out["patch_tokens"].cpu().half(),
            }, chunk_dir / f"{i:05d}.pt")
            if (i+1) % 10 == 0:
                print(f"  {dataset_name}: {(i+1)*batch_size}/{n} ({time.time()-t0:.1f}s)")
    cls_list, patch_list = [], []
    for f in sorted(chunk_dir.glob("*.pt")):
        d = torch.load(f, weights_only=False)
        cls_list.append(d["cls"])
        patch_list.append(d["patches"])
    torch.save({"cls": torch.cat(cls_list), "patches": torch.cat(patch_list)}, cache_path)
    import shutil
    shutil.rmtree(chunk_dir)
    print(f"  Done: {n} features in {time.time()-t0:.1f}s")
    return cache_path


print("=== Caching DINOv2 Features ===")
cache_paths = {}
for ds in DATASETS:
    print(f"\n{ds}:")
    cache_paths[ds] = cache_teacher_features(ds)

In [ ]:
# Cell 8: Helper Functions
def load_teacher_cache(cache_path):
    d = torch.load(cache_path, weights_only=False)
    return d["cls"].half(), d["patches"].half()


class LazyPretrainDataset(Dataset):
    def __init__(self, pil_images, t_cls, t_patches, transform):
        self.pil_images = pil_images
        self.t_cls = t_cls
        self.t_patches = t_patches
        self.transform = transform

    def __len__(self):
        return min(len(self.pil_images), len(self.t_cls))

    def __getitem__(self, idx):
        img = self.pil_images[idx]
        return self.transform(img), self.t_cls[idx], self.t_patches[idx]


def get_lazy_images(name):
    if name == "flowers102":
        raw = datasets.Flowers102(root="data", split="train", download=True)
        return [raw[i][0] for i in range(len(raw))]
    elif name == "oxford_pets":
        raw = datasets.OxfordIIITPet(root="data", download=True)
        return [raw[i][0] for i in range(len(raw))]
    elif name == "eurosat":
        raw = datasets.EuroSAT(root="data", download=True)
        return [raw[i][0] for i in range(len(raw))]
    elif name == "breastmnist":
        import medmnist
        raw = medmnist.BreastMNIST(root="data", split="train", download=True)
        images = []
        for i in range(len(raw)):
            arr = np.array(raw[i][0])
            if arr.ndim == 2:
                arr = np.stack([arr]*3, axis=-1)
            images.append(PILImage.fromarray(arr).resize((224, 224)))
        return images
    raise ValueError(f"Unknown dataset: {name}")

In [ ]:
# Cell 9: Train All Students (AMP + trainable projection)
print("=== Pre-Training TinySSL Students ===\n")
results = {}
BATCH = 32
EPOCHS = 100

for ds in DATASETS:
    num_classes = DATASET_INFO[ds]
    cache_path = cache_paths[ds]

    print(f"\nLoading {ds}...")
    t_cls, t_patches = load_teacher_cache(cache_path)
    images = get_lazy_images(ds)
    print(f"  {len(images)} images, {len(t_cls)} cached features")

    for model_name, model_cls in [("base", TinySSLBase), ("tiny", TinySSLTiny), ("cnn", TinySSLCNN)]:
        key = f"{model_name}_{ds}"
        ckpt_path = f"checkpoints/{key}.pt"

        if Path(ckpt_path).exists():
            print(f"\n--- {model_name.upper()} on {ds} --- (loading checkpoint)")
            student = model_cls().to(device)
            student.load_state_dict(torch.load(ckpt_path, weights_only=False)["model"])
            results[key] = {"model": student.cpu(), "history": {"epoch": [], "loss": [], "distill": [], "mim": []}}
            continue

        print(f"\n--- {model_name.upper()} on {ds} ---")
        student = model_cls().to(device)
        print(f"  Params: {count_params(student):.2f}M")

        dataset = LazyPretrainDataset(images, t_cls, t_patches, get_augmentation(0))
        loader = DataLoader(dataset, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)

        proj = create_projection(student.out_dim, DINO_DIM, device)
        params = list(student.parameters()) + list(proj.parameters())
        optimizer = torch.optim.AdamW(params, lr=1e-3, weight_decay=0.05)
        scaler = torch.amp.GradScaler("cuda", enabled=AMP)
        history = {"epoch": [], "loss": [], "distill": [], "mim": []}
        t0 = time.time()

        for epoch in range(EPOCHS):
            if epoch < 10:
                cur_lr = 1e-3 * (epoch + 1) / 10
            else:
                cur_lr = 1e-3 * 0.5 * (1 + math.cos(math.pi * (epoch - 10) / (EPOCHS - 10)))
            for pg in optimizer.param_groups:
                pg["lr"] = cur_lr

            mask_ratio = 0.5 if epoch < EPOCHS // 2 else 0.75
            dataset.transform = get_augmentation(epoch)
            student.train()
            proj.train()
            sum_loss, sum_d, sum_m, n_batch = 0, 0, 0, 0

            for imgs, t_cls_b, t_patches_b in loader:
                imgs = imgs.to(device)
                t_cls_b = t_cls_b.to(device)
                t_patches_b = t_patches_b.to(device)

                with torch.amp.autocast("cuda", enabled=AMP):
                    out = student(imgs)
                    s_patches = out["patches"]
                    t_interp = interp_patches(t_patches_b, s_patches.shape[1])
                    L_d = distillation_loss(s_patches, t_interp, proj)
                    B, N, D_s = s_patches.shape
                    mask = torch.rand(B, N, device=device) > mask_ratio
                    s_proj = proj(s_patches)
                    L_m = F.mse_loss(s_proj, t_interp, reduction="none")
                    L_m = L_m[mask.unsqueeze(-1).expand_as(L_m)].mean()
                    loss = L_d + 0.5 * L_m

                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                sum_loss += loss.item()
                sum_d += L_d.item()
                sum_m += L_m.item()
                n_batch += 1

            nb = max(n_batch, 1)
            history["epoch"].append(epoch)
            history["loss"].append(sum_loss / nb)
            history["distill"].append(sum_d / nb)
            history["mim"].append(sum_m / nb)

            if (epoch + 1) % 50 == 0:
                print(f"  Epoch {epoch+1}/{EPOCHS} | Loss: {sum_loss/nb:.4f} | {time.time()-t0:.1f}s")

        key = f"{model_name}_{ds}"
        results[key] = {"model": student.cpu(), "history": history}

        os.makedirs("checkpoints", exist_ok=True)
        torch.save({"model": student.state_dict(), "meta": {"model_type": model_name}},
                    f"checkpoints/{key}.pt")

    del images, t_cls, t_patches
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n=== All Students Trained ===")

In [ ]:
# Cell 9.5: Reload State from Disk (safe to re-run)
import json as _json
from pathlib import Path as _P

# Rebuild cache_paths
cache_paths = {}
for ds in DATASETS:
    p = _P("cache") / f"{ds}.pt"
    if p.exists():
        cache_paths[ds] = p

# Rebuild results dict from checkpoints
results = {}
_model_map = {"base": TinySSLBase, "tiny": TinySSLTiny, "cnn": TinySSLCNN}
for ds in DATASETS:
    for model_name, model_cls in _model_map.items():
        key = f"{model_name}_{ds}"
        ckpt = _P(f"checkpoints/{key}.pt")
        if ckpt.exists():
            student = model_cls().to(device)
            student.load_state_dict(torch.load(ckpt, weights_only=False)["model"])
            results[key] = {"model": student.cpu(), "history": {"epoch": [], "loss": [], "distill": [], "mim": []}}

# Load saved baseline/DINOv2/eval results if they exist
baseline_results = {}
if _P("baseline_results.json").exists():
    baseline_results = _json.load(open("baseline_results.json"))
    print(f"Loaded baseline_results.json ({len(baseline_results)} entries)")

dinov2_results = {}
if _P("dinov2_results.json").exists():
    dinov2_results = _json.load(open("dinov2_results.json"))
    print(f"Loaded dinov2_results.json ({len(dinov2_results)} entries)")

eval_results = {}
if _P("eval_results.json").exists():
    eval_results = _json.load(open("eval_results.json"))
    print(f"Loaded eval_results.json ({len(eval_results)} entries)")

print(f"State reloaded: {len(results)} students, {len(baseline_results)} baselines, "
      f"{len(dinov2_results)} dinov2, {len(eval_results)} evals")
for k in sorted(results.keys()):
    print(f"  {k}: checkpoint loaded")

In [ ]:
# Cell 10: Train Baselines (ResNet-18, ViT-Tiny, MobileNetV3-Small) — AMP
def train_baseline(model_name, dataset_name, num_classes, epochs=20, batch_size=64, lr=3e-4):
    train_ds, val_ds, nc = load_eval_datasets(dataset_name)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size * 2, shuffle=False, num_workers=2, pin_memory=True)

    if model_name == "resnet18":
        model = timm.create_model("resnet18", pretrained=False, num_classes=num_classes)
    elif model_name == "vit_tiny":
        model = timm.create_model("vit_tiny_patch16_224", pretrained=False, num_classes=num_classes)
    elif model_name == "mobilenetv3":
        model = timm.create_model("mobilenetv3_small_100", pretrained=False, num_classes=num_classes)
    else:
        raise ValueError(f"Unknown model: {model_name}")

    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    warmup = LinearLR(optimizer, start_factor=0.01, total_iters=min(5, epochs))
    cosine = CosineAnnealingLR(optimizer, T_max=epochs - 5, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[min(5, epochs)])

    best_acc = 0.0
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device).flatten()
            with torch.amp.autocast("cuda", enabled=AMP):
                logits = model(imgs)
                loss = criterion(logits, labels)
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * imgs.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += imgs.size(0)
        scheduler.step()

        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device).flatten()
                val_correct += (model(imgs).argmax(1) == labels).sum().item()
                val_total += labels.size(0)
        val_acc = val_correct / val_total
        if val_acc > best_acc:
            best_acc = val_acc
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs} | Loss: {total_loss/total:.4f} | Val: {val_acc:.4f} | Best: {best_acc:.4f} | {time.time()-t0:.1f}s")
    return best_acc


print("=== Training Supervised Baselines ===\n")
for ds in DATASETS:
    num_classes = DATASET_INFO[ds]
    for model_name in ["resnet18", "vit_tiny", "mobilenetv3"]:
        key = f"{model_name}_{ds}"
        if key in baseline_results:
            print(f"\n--- {model_name} on {ds} --- (already done: {baseline_results[key]['best_acc']:.4f})")
            continue
        print(f"\n--- {model_name} on {ds} ---")
        best_acc = train_baseline(model_name, ds, num_classes, epochs=20)
        baseline_results[key] = {"best_acc": best_acc}
        json.dump(baseline_results, open("baseline_results.json", "w"))
print("Baselines done.")

In [ ]:
# Cell 11: DINOv2 Linear Probe (Upper Bound) — AMP
def dinov2_linear_probe(dataset_name, num_classes, epochs=100, batch_size=64, lr=1e-3):
    train_ds, val_ds, _ = load_eval_datasets(dataset_name)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size * 2, shuffle=False, num_workers=2, pin_memory=True)

    teacher = DINOv2Teacher().to(device).eval()
    head = nn.Linear(DINO_DIM, num_classes).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    t0 = time.time()
    for epoch in range(epochs):
        head.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device).flatten()
            with torch.no_grad(), torch.amp.autocast("cuda", enabled=AMP):
                feat = teacher(imgs)["cls_token"]
            with torch.amp.autocast("cuda", enabled=AMP):
                loss = criterion(head(feat), labels)
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()

        if (epoch + 1) % 25 == 0:
            head.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for imgs, labels in val_loader:
                    imgs, labels = imgs.to(device), labels.to(device).flatten()
                    feat = teacher(imgs)["cls_token"]
                    correct += (head(feat).argmax(1) == labels).sum().item()
                    total += labels.size(0)
            print(f"  Epoch {epoch+1}/{epochs} | Val: {correct/total:.4f} | {time.time()-t0:.1f}s")

    head.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device).flatten()
            feat = teacher(imgs)["cls_token"]
            correct += (head(feat).argmax(1) == labels).sum().item()
            total += labels.size(0)
    return correct / total


print("=== DINOv2 Linear Probe (Upper Bound) ===\n")
for ds in DATASETS:
    if ds in dinov2_results:
        print(f"\n--- DINOv2 on {ds} --- (already done: {dinov2_results[ds]:.4f})")
        continue
    num_classes = DATASET_INFO[ds]
    print(f"\n--- DINOv2 on {ds} ---")
    acc = dinov2_linear_probe(ds, num_classes, epochs=100)
    dinov2_results[ds] = acc
    json.dump(dinov2_results, open("dinov2_results.json", "w"))
    print(f"  Final: {acc:.4f}")

In [ ]:
# Cell 12: Evaluate Students — Linear Probe + k-NN
def linear_probe_eval(student, dataset_name, num_classes, epochs=100, batch_size=64, lr=1e-3):
    train_ds, val_ds, _ = load_eval_datasets(dataset_name)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size * 2, shuffle=False, num_workers=2, pin_memory=True)

    student = student.to(device).eval()
    for p in student.parameters():
        p.requires_grad = False

    head = nn.Linear(student.out_dim, num_classes).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        head.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device).flatten()
            with torch.no_grad():
                feat = student(imgs)["cls"]
            loss = criterion(head(feat), labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()

    head.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device).flatten()
            feat = student(imgs)["cls"]
            correct += (head(feat).argmax(1) == labels).sum().item()
            total += labels.size(0)
    return correct / total


def knn_eval(student, dataset_name, k=20):
    train_ds, val_ds, _ = load_eval_datasets(dataset_name)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

    student = student.to(device).eval()
    for p in student.parameters():
        p.requires_grad = False

    train_feats, train_labels = [], []
    with torch.no_grad():
        for imgs, labels in train_loader:
            feat = student(imgs.to(device))["cls"]
            train_feats.append(F.normalize(feat, dim=1).cpu())
            train_labels.append(labels)
    train_feats = torch.cat(train_feats).numpy()
    train_labels = torch.cat(train_labels).numpy()

    test_feats, test_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            feat = student(imgs.to(device))["cls"]
            test_feats.append(F.normalize(feat, dim=1).cpu())
            test_labels.append(labels)
    test_feats = torch.cat(test_feats).numpy()
    test_labels = torch.cat(test_labels).numpy()

    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine", n_jobs=-1)
    knn.fit(train_feats, train_labels)
    return accuracy_score(test_labels, knn.predict(test_feats))


print("=== Evaluating All Students ===\n")
for ds in DATASETS:
    num_classes = DATASET_INFO[ds]
    for model_name in ["base", "tiny", "cnn"]:
        key = f"{model_name}_{ds}"
        if key not in results:
            continue
        if key in eval_results:
            print(f"\n--- {model_name.upper()} on {ds} --- (already done: LP={eval_results[key]['linear_probe']:.4f})")
            continue
        student = results[key]["model"]
        print(f"\n--- {model_name.upper()} on {ds} ---")
        lp_acc = linear_probe_eval(student, ds, num_classes, epochs=100)
        knn_acc = knn_eval(student, ds)
        eval_results[key] = {"linear_probe": lp_acc, "knn": knn_acc, "params": count_params(student)}
        json.dump(eval_results, open("eval_results.json", "w"))
        print(f"  Linear Probe: {lp_acc:.4f} | k-NN: {knn_acc:.4f}")

In [ ]:
# Cell 13: Results Table
print("\n" + "=" * 90)
print("TINYSSL RESULTS — Knowledge Distillation from DINOv2")
print("=" * 90)
header = f"{'Method':<25} {'Params':>7} {'Flowers':>8} {'Pets':>8} {'EuroSAT':>8} {'BreastMN':>9}"
print(header)
print("-" * 90)

for model_name in ["base", "tiny", "cnn"]:
    params = eval_results.get(f"{model_name}_flowers102", {}).get("params", 0)
    flowers = eval_results.get(f"{model_name}_flowers102", {}).get("linear_probe", 0)
    pets = eval_results.get(f"{model_name}_oxford_pets", {}).get("linear_probe", 0)
    eurosat = eval_results.get(f"{model_name}_eurosat", {}).get("linear_probe", 0)
    breast = eval_results.get(f"{model_name}_breastmnist", {}).get("linear_probe", 0)
    label = f"TinySSL-{model_name.capitalize()}"
    print(f"{label:<25} {params:>5.1f}M {flowers:>7.1%} {pets:>7.1%} {eurosat:>7.1%} {breast:>8.1%}")

print("-" * 90)
for bl_name in ["resnet18", "vit_tiny", "mobilenetv3"]:
    flowers = baseline_results.get(f"{bl_name}_flowers102", {}).get("best_acc", 0)
    pets = baseline_results.get(f"{bl_name}_oxford_pets", {}).get("best_acc", 0)
    eurosat = baseline_results.get(f"{bl_name}_eurosat", {}).get("best_acc", 0)
    breast = baseline_results.get(f"{bl_name}_breastmnist", {}).get("best_acc", 0)
    label = bl_name.upper().replace("_", "-")
    print(f"{label:<25} {'--':>7} {flowers:>7.1%} {pets:>7.1%} {eurosat:>7.1%} {breast:>8.1%}")

print("-" * 90)
flowers = dinov2_results.get("flowers102", 0)
pets = dinov2_results.get("oxford_pets", 0)
eurosat = dinov2_results.get("eurosat", 0)
breast = dinov2_results.get("breastmnist", 0)
print(f"{'DINOv2-S/14 (linear)':<25} {'22.0M':>7} {flowers:>7.1%} {pets:>7.1%} {eurosat:>7.1%} {breast:>8.1%}")
print("=" * 90)

In [ ]:
# Cell 14: Visualizations
os.makedirs("figures", exist_ok=True)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for idx, ds in enumerate(DATASETS):
    ax = axes[idx]
    key = f"base_{ds}"
    if key in results:
        h = results[key]["history"]
        ax.plot(h["epoch"], h["loss"], label="Total", linewidth=2)
        ax.plot(h["epoch"], h["distill"], label="Distill", linewidth=1.5, alpha=0.8)
        ax.plot(h["epoch"], h["mim"], label="MIM", linewidth=1.5, alpha=0.8)
    ax.set_title(ds.replace("_", " ").title(), fontsize=14)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.suptitle("TinySSL-Base Training Curves", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig("figures/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(DATASETS))
width = 0.15
methods = [("TinySSL-Base", "base"), ("TinySSL-Tiny", "tiny"), ("ResNet-18", "resnet18"),
           ("ViT-Tiny", "vit_tiny"), ("MobileNetV3-S", "mobilenetv3"), ("DINOv2", "dinov2")]
colors = ["#2196F3", "#64B5F6", "#FF9800", "#FFB74D", "#4CAF50", "#F44336"]
for i, (label, tag) in enumerate(methods):
    vals = []
    for ds in DATASETS:
        if tag == "dinov2":
            vals.append(dinov2_results.get(ds, 0) * 100)
        elif tag in ("resnet18", "vit_tiny", "mobilenetv3"):
            vals.append(baseline_results.get(f"{tag}_{ds}", {}).get("best_acc", 0) * 100)
        else:
            vals.append(eval_results.get(f"{tag}_{ds}", {}).get("linear_probe", 0) * 100)
    ax.bar(x + i * width, vals, width, label=label, color=colors[i])
ax.set_ylabel("Linear Probe Accuracy (%)", fontsize=12)
ax.set_title("Method Comparison Across Domains", fontsize=16)
ax.set_xticks(x + width * 2.5)
ax.set_xticklabels([d.replace("_", " ").title() for d in DATASETS], fontsize=11)
ax.legend(fontsize=9, loc="lower right")
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig("figures/comparison_bars.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
for ds_idx, ds in enumerate(DATASETS):
    color = ["#E91E63", "#9C27B0", "#3F51B5", "#009688"][ds_idx]
    for model_name in ["base", "tiny", "cnn"]:
        key = f"{model_name}_{ds}"
        if key in eval_results:
            p = eval_results[key]["params"]
            a = eval_results[key]["linear_probe"] * 100
            ax.scatter(p, a, c=color, s=100, zorder=5, edgecolors="black", linewidth=0.5)
    ax.scatter(22, dinov2_results.get(ds, 0) * 100, c=color, s=150, marker="*", zorder=5,
               edgecolors="black", linewidth=0.5)
for ds_idx, ds in enumerate(DATASETS):
    color = ["#E91E63", "#9C27B0", "#3F51B5", "#009688"][ds_idx]
    key = f"base_{ds}"
    if key in eval_results:
        ax.annotate(ds.replace("_", " ").title(),
                     (eval_results[key]["params"], eval_results[key]["linear_probe"] * 100),
                     textcoords="offset points", xytext=(5, 5), fontsize=9, color=color)
ax.set_xlabel("Parameters (M)", fontsize=12)
ax.set_ylabel("Linear Probe Accuracy (%)", fontsize=12)
ax.set_title("Efficiency: Params vs Accuracy", fontsize=16)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/params_vs_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for col, ds in enumerate(DATASETS):
    train_ds, _, _ = load_eval_datasets(ds)
    img, label = train_ds[0]
    if isinstance(img, torch.Tensor):
        img_show = img.permute(1, 2, 0).numpy()
        img_show = (img_show - img_show.min()) / (img_show.max() - img_show.min())
    else:
        img_show = np.array(img)
    axes[col].imshow(img_show)
    axes[col].set_title(f"{ds.replace('_',' ').title()}\nClass: {label}", fontsize=11)
    axes[col].axis("off")
plt.suptitle("Sample Images from Each Domain", fontsize=16, y=1.05)
plt.tight_layout()
plt.savefig("figures/sample_images.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 15: Save Results
all_results = {
    "dinov2": dinov2_results,
    "students": {k: {kk: vv for kk, vv in v.items() if kk != "model"} for k, v in results.items()},
    "baselines": baseline_results,
    "eval": eval_results,
}

for k in all_results["students"]:
    if "history" in all_results["students"][k]:
        h = all_results["students"][k]["history"]
        all_results["students"][k]["history"] = {kk: [float(x) for x in vv] for kk, vv in h.items()}

with open("results.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("Results saved to results.json")
print("Figures saved to figures/")

print("\n" + "=" * 60)
print("KEY TAKEAWAYS")
print("=" * 60)
base_flowers = eval_results.get("base_flowers102", {}).get("linear_probe", 0)
dinov2_flowers = dinov2_results.get("flowers102", 0)
if dinov2_flowers > 0:
    ratio = base_flowers / dinov2_flowers * 100
    print(f"TinySSL-Base retains {ratio:.0f}% of DINOv2's accuracy on Flowers-102")
    if "base_flowers102" in results:
        print(f"  with {count_params(results['base_flowers102']['model']):.1f}M params vs 22M (DINOv2-S/14)")
print("=" * 60)

In [ ]:
# Cell 16: Upload to Hugging Face Hub

!pip install -q huggingface_hub
from huggingface_hub import HfApi, login
import os

HF_TOKEN = "hf_YOUR_TOKEN_HERE"  # Replace with your Hugging Face token
login(token=HF_TOKEN)
api = HfApi()
REPO = "emran696996966/TinySSL"  # <-- change this to your HF username

api.create_repo(repo_id=REPO, repo_type="model", exist_ok=True)
print(f"Uploading to {REPO}...")

import glob
for f in sorted(glob.glob("checkpoints/*.pt")):
    print(f"  Uploading {f}...")
    api.upload_file(path_or_fileobj=f, path_in_repo=f"checkpoints/{os.path.basename(f)}", repo_id=REPO)

for f in ["results.json", "baseline_results.json", "dinov2_results.json", "eval_results.json"]:
    if os.path.exists(f):
        print(f"  Uploading {f}...")
        api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=REPO)

for f in sorted(glob.glob("figures/*.png")):
    print(f"  Uploading {f}...")
    api.upload_file(path_or_fileobj=f, path_in_repo=f"figures/{os.path.basename(f)}", repo_id=REPO)

print(f"Done! View at: https://huggingface.co/{REPO}")